In [2]:
import fitz
import numpy as np
import json
import os
from litellm import completion, embedding

# plain openai also can be used
# from openai import OpenAI

# initilize openai client
# client = OpenAI(,
#     api_key=os.getenv("OPENAI_API_KEY")  # Retrieve the API key from environment variables
# )

# we are using litellm as it allows us to easily switch between different LLM providers
# and is compatible with the same API

# Configure API keys (replace with your actual keys)



In [3]:


def extract_text_from_pdf(pdf_path):
    """
    Extracts and consolidates text from all pages of a PDF file. This is the first step in the RAG pipeline,
    where we acquire the raw textual data that will later be processed, embedded, and retrieved against.

    Args:
    pdf_path (str): Path to the PDF file to be processed.

    Returns:
    str: Complete extracted text from all pages of the PDF, concatenated into a single string.
         This raw text will be further processed in subsequent steps of the RAG pipeline.
    """
    # Open the PDF file
    mypdf = fitz.open(pdf_path)
    all_text = ""  # Initialize an empty string to store the extracted text

    # Iterate through each page in the PDF
    for page_num in range(mypdf.page_count):
        page = mypdf[page_num]  # Get the page
        text = page.get_text("text")  # Extract text from the page
        all_text += text  # Append the extracted text to the all_text string

    return all_text  # Return the extracted text


In [4]:
def chunk_text(text, n, overlap):
    """
    Divides text into smaller, overlapping chunks for more effective processing and retrieval.
    Chunking is a critical step in RAG systems as it:
    1. Makes large documents manageable for embedding models that have token limits
    2. Enables more precise retrieval of relevant information
    3. Allows for contextual understanding within reasonable boundaries
    
    The overlap between chunks helps maintain context continuity and reduces the risk of
    splitting important information across chunk boundaries.

    Args:
    text (str): The complete text to be chunked.
    n (int): The maximum number of characters in each chunk.
    overlap (int): The number of overlapping characters between consecutive chunks.
                   Higher overlap improves context preservation but increases redundancy.

    Returns:
    List[str]: A list of text chunks that will be individually embedded and used for retrieval.
    """
    chunks = []  # Initialize an empty list to store the chunks
    
    # Loop through the text with a step size of (n - overlap)
    for i in range(0, len(text), n - overlap):
        # Append a chunk of text from index i to i + n to the chunks list
        chunks.append(text[i:i + n])

    return chunks  # Return the list of text chunks

In [5]:
pdf_path = "data/Epstein.pdf"

# Extract text from the PDF file
extracted_text = extract_text_from_pdf(pdf_path)

# Chunk the extracted text into segments of 1000 characters with an overlap of 200 characters
text_chunks = chunk_text(extracted_text, 1000, 200)

# Print the number of text chunks created
print("Number of text chunks:", len(text_chunks))

# Print the first text chunk
print("\nFirst text chunk:")
print(text_chunks[0])

Number of text chunks: 21

First text chunk:
1/11
Hugh Dougherty
Listen To The Jeffrey Epstein Tapes: ‘I Was Donald
Trump’s Closest Friend’
thedailybeast.com/listen-to-the-jeffrey-epstein-tapes-i-was-donald-trumps-closest-friend
Elections
THE PREDATOR AND THE PRESIDENT
Explosive tapes recorded by author Michael Wolff show Epstein
claiming Trump liked to “f---” his friends’ wives and first slept with
Melania on the “Lolita Express.”
Exclusive
Photo Illustration by Thomas Levinson/The Daily Beast/Getty Images
Jeffrey Epstein described himself as Donald Trump’s “closest friend” and claimed intimate
knowledge of his proclivity for sex, including cuckolding his best friends, according to
recordings obtained exclusively by the Daily Beast.
The convicted pedophile even boasted of his closeness to Trump and his now-wife Melania
by claiming, “the first time he slept with her was on my plane,” which was dubbed the Lolita
Express.
Epstein spoke at length about Trump with the author Michael Wolff 

In [7]:
def create_embeddings(text, model="huggingface/sentence-transformers/all-MiniLM-L6-v2"):
    """
    Transforms text into dense vector representations (embeddings) using a neural network model.
    Embeddings are the cornerstone of modern RAG systems because they:
    1. Capture semantic meaning in a numerical format that computers can process
    2. Enable similarity-based retrieval beyond simple keyword matching
    3. Allow for efficient indexing and searching of large document collections
    
    In RAG, both document chunks and user queries are embedded in the same vector space,
    allowing us to find the most semantically relevant chunks for a given query.

    Args:
    text (str or List[str]): The input text(s) to be embedded. Can be a single string or a list of strings.
    model (str): The embedding model to use. Default is OpenAI's "text-embedding-ada-002".
                 Different models offer various tradeoffs between quality, speed, and cost.

    Returns:
    dict: The response from the API containing the embeddings, which are high-dimensional
          vectors representing the semantic content of the input text(s).
    """
    # Create embeddings for the input text using the specified model
    response = embedding(model=model, input=text,api_key=os.environ.get("HUGGINGFACE_API_KEY"))

    return response  # Return the response containing the embeddings

# Create embeddings for the text chunks
response = create_embeddings(text_chunks)


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



APIError: litellm.APIError: HuggingfaceException - Not Found